# EEEM068 - Group 13 | Book 1 - Main Training Scenarios
## Setup


Ensure you have connected to a T4 runtime

Checkout project from main git branch and install requirements

In [ ]:
import os
from google.colab import drive

branch = "main"
!git clone --depth 1 -b {branch} https://github.com/monezamipoor/melanoma_classification.git
%cd melanoma_classification
!pip install -r requirements.txt

Download the training and test data and extract.

In [ ]:
import gdown
import zipfile
import os

# 1. Google Drive file ID or shareable link
file_id = '1tcXcfGPwVjzPccKHVec-AcXIibwpddUV'
url = f'https://drive.google.com/uc?id={file_id}'

# 2. Download the file
zip_path = 'file.zip'
gdown.download(url, zip_path, quiet=True)

# 3. Unzip the file
unzip_dir = './data'
os.makedirs(unzip_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for member in zip_ref.infolist():
        # Construct the full path where this file would be extracted
        extracted_path = os.path.join(unzip_dir, member.filename)

        # Skip if the file already exists
        if os.path.exists(extracted_path):
            print(f"Skipping existing file: {member.filename}")
            continue
        else:
            print(f"Extracting: {member.filename}")

        # Make sure any required directories exist
        os.makedirs(os.path.dirname(extracted_path), exist_ok=True)

        # Extract this file only
        with zip_ref.open(member) as source, open(extracted_path, 'wb') as target:
            target.write(source.read())

print("Extraction complete")

# 1. Optimizing the baseline

## 1. Baseline model

In [ ]:
!python main.py --opt 1-Base.yml

## 2. With tuned hyperparameters for training

In [ ]:
!python main.py --opt 2-Hyperparameters.yml

## 3. With tuned hyperparameters for trainingoptimized oversampling and undersampling strategies

In [ ]:
!python main.py --opt 3-Sampling.yml

## 4. With augmentations added

In [ ]:
!python main.py --opt 4-Augmnetation.yml

## 5. With loss functions tuned

In [ ]:
!python main.py --opt 5-Loss.yml

## 6. Testing models with these parameters tuned

In [ ]:
!python main.py --opt 6-Model.yml

## 7. with K-Fold ensemble added

In [ ]:
!python main.py --opt 7-Kfold.yml

# 2. Extra models added

## 1. Hybrid CNN/Transformer model

This one is control test

In [ ]:
!python main.py --opt 8-a-Hybrid.yml

This one is hybrid model added

In [ ]:
!python main.py --opt 8-b-Hybrid.yml

## 2. Using metadata

In [ ]:
!python main.py --opt 9-Metadata.yml

## 3. Contrastive pretraining

In [ ]:
!python main.py --opt 10-Contrastive.yml

## 4. Explainable AI
### Grad-CAM++, ScoreCAM, KPCA-CAM, and Finer CAM

##💡 <font color="#6425d0">Configure XAI for Your Melanoma Skin Disease Model</font>

- <b>model_pth</b> -> Its the location of our .pth file
- <b>image_path</b> -> Its the location of our targeted test image

In [ ]:
# model set
model_pth="/content/2025-04-29_00-58-54-NF_E4-model_resnet50_AUC.pth"
image_path="/content/Melanoma/test/ISIC_0822581.jpg"

## 1. Imports and Setup

In [ ]:
!pip install grad-cam

In [ ]:
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.models as models
import torchvision.transforms as T
from collections import OrderedDict

# Explainable AI tools
from pytorch_grad_cam import (
    GradCAMPlusPlus,
    ScoreCAM,
    KPCA_CAM,
    FinerCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

This cell prepares our compute device, data preprocessing pipeline, model architecture, and loads our trained weights so we can immediately run inference or XAI.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std= [0.229, 0.224, 0.225]),
])

model = models.resnet50(pretrained=False)
in_feats = model.fc.in_features
model.fc = torch.nn.Linear(in_feats, 1)

# Load & unwrap checkpoint
raw = torch.load(model_pth, map_location="cpu")

This cell renames the keys in our saved checkpoint so they line up with our torchvision model’s layers, then loads them in.

In [ ]:
# Clean up the loaded state dict so its keys match torchvision’s naming
cleaned_state_dict = OrderedDict()
for original_key, weight_tensor in raw.items():
    new_key = original_key
    if new_key.startswith("backbone."):
        new_key = new_key[len("backbone."):]
    if new_key.startswith("model."):
        new_key = new_key[len("model."):]
    cleaned_state_dict[new_key] = weight_tensor

# Load into model
missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)
model.to(device).eval()

Our XAI function that loads and preprocesses an input image, runs it through our model, generates multiple class activation maps (e.g., Grad-CAM++, ScoreCAM) for a specified layer, and then displays each heatmap alongside the original image.

In [ ]:
def XAI(image_path, model, transform):
  # 1) Load & preprocess image for the model
    orig = Image.open(image_path).convert('RGB')
    inp  = transform(orig).unsqueeze(0).to(device)

  # 2) Run inference to get logits and pick the class to explain
    with torch.no_grad():
        logits = model(inp)
        probs  = torch.sigmoid(logits)
        top_cl = 0

  # 3) Specify the layer to probe and assemble CAM methods
    target_layer = model.layer4[-1].conv3
    cam_algos = {
        "Grad-CAM++":   GradCAMPlusPlus(model=model, target_layers=[target_layer]),
        "ScoreCAM":     ScoreCAM(model=model, target_layers=[target_layer]),
        "KPCA-CAM":     KPCA_CAM(model=model, target_layers=[target_layer]),
        "FinerCAM":     FinerCAM(model=model, target_layers=[target_layer]),
    }

  # 4) Generate & display each CAM alongside the original image
    results = [("Original", orig)]
    rgb_np  = np.array(orig.resize((224,224))).astype(np.float32) / 255.0

    for title, cam_obj in cam_algos.items():
        cam_map = cam_obj(inp, targets=[ClassifierOutputTarget(top_cl)])[0]
        cam_img = show_cam_on_image(rgb_np, cam_map, use_rgb=True)
        results.append((title, Image.fromarray(cam_img)))

    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5))
    for ax, (title, img) in zip(axes, results):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
XAI(image_path, model, transform)